# Cryptic IP Binding Sites - Full Pipeline

This notebook provides a complete environment to run the Cryptic IP phase validation and proteome screening pipeline. It installs all dependencies directly into the Google Colab environment.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_ORG/cryptic-ip-binding-sites/blob/main/notebooks/CrypticIP_FullPipeline.ipynb)

## SECTION 1 — Environment Setup

In [ ]:
# Install Python packages
!pip install freesasa biopython pandas numpy matplotlib seaborn requests py3Dmol ipywidgets tqdm pytest

# Install System Tools (fpocket, apbs, pdb2pqr)
!apt-get update -qq
!apt-get install -y -qq fpocket apbs pdb2pqr > /dev/null

# Verify installations
!fpocket -v
!apbs --version || true
!pdb2pqr --version

In [ ]:
import os
import sys
from pathlib import Path

# Clone the repository if not already present
if not os.path.exists('cryptic-ip-binding-sites'):
    # Note: Replace with actual GitHub URL when published. Using local clone block for now if already mapped.
    !git clone https://github.com/example/cryptic-ip-binding-sites.git
    
if os.path.exists('cryptic-ip-binding-sites'):
    os.chdir('cryptic-ip-binding-sites')
sys.path.append(os.getcwd())

import requests
import py3Dmol
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import ipywidgets as widgets
from IPython.display import display, HTML
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Environment ready!")

## SECTION 2 — Phase 1: Validation (ADAR2)
We validate the pipeline on our gold-standard positive control: ADAR2 (PDB 1ZY7 / AF-P78563).

In [ ]:
from pipeline.utils import load_config
from validation.validation_report import process_protein

# Download structures
val_dir = Path("data/structures/validation")
val_dir.mkdir(parents=True, exist_ok=True)
out_dir = Path("data/results/validation")
out_dir.mkdir(parents=True, exist_ok=True)

adar_af_path = val_dir / "AF-P78563-F1-model_v4.pdb"
if not adar_af_path.exists():
    print("Downloading ADAR2 AlphaFold...")
    response = requests.get("https://alphafold.ebi.ac.uk/files/AF-P78563-F1-model_v4.pdb")
    with open(adar_af_path, "wb") as f:
        f.write(response.content)

print("Running pipeline on ADAR2...")
adar_pockets = process_protein(adar_af_path, out_dir)

print(f"Found {len(adar_pockets)} valid pockets.")
df_adar = pd.DataFrame(adar_pockets)
if not df_adar.empty:
    cols = ['id', 'composite_score', 'mean_sasa', 'electrostatic_potential', 'basic_residue_count', 'mean_plddt', 'pass_all_filters']
    display(df_adar[[c for c in cols if c in df_adar.columns]].head())

# Output GO/NO-GO logic cell for visual reinforcement
config = load_config()
adar_pass = False
if not df_adar.empty:
    # Check top N pockets
    top_n = config["pipeline"]["validation"]["adar2_max_pocket_rank"]
    top_pockets = df_adar.head(top_n)
    if any(top_pockets['pass_all_filters']):
        adar_pass = True

if adar_pass:
    print("\n✅ GO: ADAR2 passed validation gating criteria.")
else:
    print("\n❌ NO-GO: ADAR2 failed basic evaluation target validation.")

In [ ]:
if not df_adar.empty:
    best_pocket_id = df_adar.iloc[0]['id']
    pkt_pdb = out_dir / f"AF-P78563-F1-model_v4_out/pockets/pocket{best_pocket_id}_atm.pdb"
    
    if pkt_pdb.exists():
        print(f"Visualizing best pocket (ID: {best_pocket_id})")
        view = py3Dmol.view(width=800, height=600)
        # Load main protein
        with open(adar_af_path, 'r') as f:
            view.addModel(f.read(), 'pdb')
        view.setStyle({'cartoon': {'color': '#ECECEC'}})
        
        # Highlight pocket
        with open(pkt_pdb, 'r') as f:
            view.addModel(f.read(), 'pdb')
        view.setStyle({'model': 1}, {'sphere': {'color': 'red', 'radius': 1.0}})
        
        view.zoomTo({'model': 1})
        view.show()
    else:
        print("Pocket PDB not generated (fpocket may not be installed or enabled correctly in Colab).")

## SECTION 3 — Extended Validation
Run all positive and negative controls to plot and inspect the score separability.

In [ ]:
print("Downloading Validation Controls...")
controls = [
    ("1ZY7", "positive"), ("5HDT", "positive"), ("5ICN", "positive"), ("4A69", "positive"),
    ("1MAI", "negative"), ("1BTK", "negative"), ("1FAO", "negative"), ("1FGY", "negative")
]

results = []
for pdb_id, ctype in tqdm(controls):
    path = val_dir / f"{pdb_id}.pdb"
    if not path.exists():
        response = requests.get(f"https://files.rcsb.org/download/{pdb_id}.pdb")
        with open(path, "wb") as f:
            f.write(response.content)
    
    pockets = process_protein(path, out_dir)
    top_score = pockets[0]['composite_score'] if pockets else 0.0
    results.append({'PDB': pdb_id, 'Type': ctype, 'TopScore': top_score})

df_val = pd.DataFrame(results)
display(df_val)

In [ ]:
plt.figure(figsize=(10, 6))
sns.violinplot(data=df_val, x='Type', y='TopScore', inner="point", cut=0)
plt.axhline(config["pipeline"]["scoring"]["min_composite_score_for_flag"], color='red', linestyle='--', label='Candidate Threshold')
plt.axhline(config["pipeline"]["validation"]["negative_control_max_score"], color='orange', linestyle=':', label='Negative Control Limit')
plt.title("Control Class Separability in Composite Score (Violin Plot)")
plt.legend()
plt.show()

# Check gating boundary overlap
min_pos = df_val[df_val['Type'] == 'positive']['TopScore'].min()
max_neg = df_val[df_val['Type'] == 'negative']['TopScore'].max()
if min_pos > max_neg:
    print("\n✅ GO: Positive and negative controls exhibit clear separability without overlap.")
else:
    print("\n❌ NO-GO: Score distributions of controls overlap.")

## SECTION 4 — Yeast Proteome Screen (optional, resource-heavy)
⚠️ **Warning**: This section downloads ~15 GB and takes ~6 hours on Colab CPU. Consider using GitHub Actions instead. We implement chunked downloading with `tqdm` below.

In [ ]:
import tarfile
import tempfile

config = load_config()
yeast_url = config["organisms"]["yeast"]["download_url"]
yeast_dir = Path("data/structures/yeast")

# Toggle `run_yeast_screen` to True to execute the massive pipeline on Colab CPU
run_yeast_screen = False 

def download_file_with_tqdm(url, dest_path):
    response = requests.get(url, stream=True)
    total_size = int(response.headers.get('content-length', 0))
    block_size = 1024 * 1024 # 1 MB
    
    with open(dest_path, 'wb') as file, tqdm(desc=dest_path.name, total=total_size, unit='iB', unit_scale=True, unit_divisor=1024) as bar:
        for data in response.iter_content(block_size):
            size = file.write(data)
            bar.update(size)

if run_yeast_screen:
    # Chunked download with exact progress bar mapping
    yeast_dir.mkdir(parents=True, exist_ok=True)
    tar_path = yeast_dir / "yeast_proteome.tar"
    if not tar_path.exists():
        print("Downloading full yeast AlphaFold proteome (chunked)...")
        download_file_with_tqdm(yeast_url, tar_path)
        
    # Extraction logic
    extracted_count = len(list(yeast_dir.glob("*.pdb")))
    expected_count = config["organisms"]["yeast"]["expected_file_count"]
    
    if extracted_count != expected_count:
        print(f"Extracting tar architecture for {expected_count} structures...")
        with tarfile.open(tar_path, "r:") as tar:
            tar.extractall(path=yeast_dir)
        print("Extraction complete.")
        
    # Run screen_proteome script completely (multiprocessed)
    print("Running proteome screen array...")
    !python -m screening.screen_proteome --organism=yeast --workers=2 --batch-size=500
    !python -m screening.triage --organism=yeast
else:
    print("Yeast screen skipped. Set `run_yeast_screen = True` to execute.")

## SECTION 5 — Results Analysis
Visualizing the results for target triage.

In [ ]:
# Load actual results if running screen, else we load from the filesystem if present
results_dir = Path("data/results/yeast")
csv_path = results_dir / "yeast_master.csv"

if csv_path.exists():
    df_hits = pd.read_csv(csv_path)
else:
    print("Master dataset CSV not found. For this notebook representation we will load a mock synthetic dataframe modeling what you would expect from the output generation.")
    # Mock data to ensure the rest of the notebook visualization runs for demo without the 6 hour pipeline output.
    mock_data = [
        {"UniProtID": "P12345", "Organism": "yeast", "ProteinName": "Mock_Hit_1", "PocketID": 1, "CompositeScore": 0.85, "PocketVolume": 450.0, "MeanSASA": 2.3, "ElectrostaticPotential": 12.0, "BasicResidueCount": 5, "MeanPLDDT": 92.0, "PassAllFilters": True, "ManualReviewFlag": True},
        {"UniProtID": "Q98765", "Organism": "yeast", "ProteinName": "Mock_Hit_2", "PocketID": 3, "CompositeScore": 0.72, "PocketVolume": 320.0, "MeanSASA": 15.1, "ElectrostaticPotential": 6.5, "BasicResidueCount": 4, "MeanPLDDT": 87.5, "PassAllFilters": True, "ManualReviewFlag": True},
        {"UniProtID": "A1B2C3", "Organism": "human", "ProteinName": "Mock_Human_1", "PocketID": 2, "CompositeScore": 0.91, "PocketVolume": 610.0, "MeanSASA": 0.1, "ElectrostaticPotential": 18.2, "BasicResidueCount": 7, "MeanPLDDT": 98.1, "PassAllFilters": True, "ManualReviewFlag": True}
    ]
    for _ in range(200): # Fill with generic background noise
        mock_data.append({"UniProtID": f"NOISE_{_}", "Organism": "yeast", "ProteinName": "Noise", "CompositeScore": np.random.uniform(0.1, 0.6), "PocketVolume": np.random.uniform(200, 1000), "MeanSASA": np.random.uniform(20, 100), "ElectrostaticPotential": np.random.uniform(-10, 5), "BasicResidueCount": np.random.randint(0, 3), "MeanPLDDT": np.random.uniform(40, 90), "ManualReviewFlag": False})
    df_hits = pd.DataFrame(mock_data)
    
flagged_candidates = df_hits[df_hits['ManualReviewFlag'] == True]

# 1. Comparative hit rate visualization (Bar chart)
hit_counts = df_hits.groupby('Organism')['ManualReviewFlag'].sum().reset_index()
hit_counts.rename(columns={'ManualReviewFlag': 'CandidateCount'}, inplace=True)

plt.figure(figsize=(6, 4))
sns.barplot(data=hit_counts, x='Organism', y='CandidateCount', palette='pastel')
plt.title('Total Candidates per Proteome')
plt.ylabel('Count')
plt.show()

# 2. Scatter plot: composite score vs pLDDT for all candidates
plt.figure(figsize=(8, 5))
sns.scatterplot(data=flagged_candidates, x='MeanPLDDT', y='CompositeScore', hue='MeanSASA', size='PocketVolume', sizes=(50, 200), palette='coolwarm')
plt.title('Score vs pLDDT (Top Candidates)')
plt.legend(bbox_to_anchor=(1.05, 1), loc=2)
plt.show()

# 3. Heatmap of key features
top_50 = flagged_candidates.sort_values(by='CompositeScore', ascending=False).head(50)
if not top_50.empty:
    heatmap_data = top_50[['MeanSASA', 'ElectrostaticPotential', 'BasicResidueCount', 'MeanPLDDT']].copy()
    # Normalize columns for heatmap
    heatmap_data = (heatmap_data - heatmap_data.min()) / (heatmap_data.max() - heatmap_data.min())
    heatmap_data.index = top_50['UniProtID']
    
    plt.figure(figsize=(8, 10))
    sns.heatmap(heatmap_data, cmap='viridis', annot=False)
    plt.title('Normalized Feature Profile of Top 50 Screened Candidates')
    plt.show()

## SECTION 6 — Hit Triage
Interactive table and active links for manual structural and functional review.

In [ ]:
def format_links(val):
    af_link = f'<a target="_blank" href="https://alphafold.ebi.ac.uk/entry/{val}">AF Pattern</a>'
    up_link = f'<a target="_blank" href="https://www.uniprot.org/uniprot/{val}">UniProt</a>'
    return f"{af_link} | {up_link}"

# Add links to table dataframe
if not top_50.empty:
    triage_table = top_50[['UniProtID', 'Organism', 'CompositeScore', 'MeanSASA', 'BasicResidueCount', 'MeanPLDDT']].copy()
    triage_table.insert(1, "Inspection", triage_table['UniProtID'].apply(format_links))
    
    display(HTML(triage_table.to_html(escape=False, index=False)))
    
    # Provide export capability locally or in Colab
    out_csv = "top_triage_candidates.csv"
    top_50.to_csv(out_csv, index=False)
    
    if IN_COLAB:
        print(f"\nExporting {out_csv} via browser download...")
        files.download(out_csv)
    else:
        print(f"\nSaved candidates triage table to {out_csv}")
else:
    print("No candidates detected passing thresholds.")